In [1]:
import pandas as pd
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from collections import defaultdict
from astropy.coordinates import SkyCoord
import astropy.units as u
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)

In [2]:
# Loading Data
galaxyzoo = Table.read("/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/gz_desi_deep_learning_catalog_friendly.csv")
galaxyzoo

dr8_id,ra,dec,brickid,objid,smooth-or-featured_smooth_fraction,smooth-or-featured_featured-or-disk_fraction,smooth-or-featured_artifact_fraction,disk-edge-on_yes_fraction,disk-edge-on_no_fraction,has-spiral-arms_yes_fraction,has-spiral-arms_no_fraction,bar_strong_fraction,bar_weak_fraction,bar_no_fraction,bulge-size_dominant_fraction,bulge-size_large_fraction,bulge-size_moderate_fraction,bulge-size_small_fraction,bulge-size_none_fraction,how-rounded_round_fraction,how-rounded_in-between_fraction,how-rounded_cigar-shaped_fraction,edge-on-bulge_boxy_fraction,edge-on-bulge_none_fraction,edge-on-bulge_rounded_fraction,spiral-winding_tight_fraction,spiral-winding_medium_fraction,spiral-winding_loose_fraction,spiral-arm-count_1_fraction,spiral-arm-count_2_fraction,spiral-arm-count_3_fraction,spiral-arm-count_4_fraction,spiral-arm-count_more-than-4_fraction,spiral-arm-count_cant-tell_fraction,merging_none_fraction,merging_minor-disturbance_fraction,merging_major-disturbance_fraction,merging_merger_fraction,catalog_version,legacy_survey_data_release
str12,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str5,str3
100000_1081,32.084931,-44.311422,100000,1081,0.69,0.25,0.06,--,--,--,--,--,--,--,--,--,--,--,--,0.029999999329447746,0.5400000214576721,0.4399999976158142,--,--,--,--,--,--,--,--,--,--,--,--,0.84,0.12,0.02,0.01,1.0.0,DR8
100000_1401,32.140085,-44.293668,100000,1401,0.77,0.12,0.11,--,--,--,--,--,--,--,--,--,--,--,--,0.8799999952316284,0.10999999940395355,0.009999999776482582,--,--,--,--,--,--,--,--,--,--,--,--,0.6,0.15,0.05,0.21,1.0.0,DR8
100000_1483,32.275015,-44.288957,100000,1483,0.81,0.1,0.08,--,--,--,--,--,--,--,--,--,--,--,--,0.07000000029802322,0.8799999952316284,0.05999999865889549,--,--,--,--,--,--,--,--,--,--,--,--,0.59,0.17,0.04,0.19,1.0.0,DR8
100000_1509,32.045648,-44.287172,100000,1509,0.64,0.27,0.09,--,--,--,--,--,--,--,--,--,--,--,--,0.38999998569488525,0.5400000214576721,0.07999999821186066,--,--,--,--,--,--,--,--,--,--,--,--,0.17,0.07,0.05,0.71,1.0.0,DR8
100000_1869,32.170627,-44.267273,100000,1869,0.88,0.05,0.07,--,--,--,--,--,--,--,--,--,--,--,--,0.019999999552965164,0.8199999928474426,0.1599999964237213,--,--,--,--,--,--,--,--,--,--,--,--,0.68,0.25,0.05,0.02,1.0.0,DR8
100000_1901,32.264953,-44.264718,100000,1901,0.32,0.48,0.2,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,0.56,0.2,0.21,0.02,1.0.0,DR8
100000_2004,32.104085,-44.257405,100000,2004,0.86,0.07,0.07,--,--,--,--,--,--,--,--,--,--,--,--,0.019999999552965164,0.8500000238418579,0.12999999523162842,--,--,--,--,--,--,--,--,--,--,--,--,0.61,0.31,0.06,0.02,1.0.0,DR8
100000_2198,32.097469,-44.250979,100000,2198,0.86,0.06,0.09,--,--,--,--,--,--,--,--,--,--,--,--,0.3400000035762787,0.6499999761581421,0.009999999776482582,--,--,--,--,--,--,--,--,--,--,--,--,0.88,0.09,0.02,0.01,1.0.0,DR8
100000_2211,32.149425,-44.242194,100000,2211,0.72,0.11,0.17,--,--,--,--,--,--,--,--,--,--,--,--,0.7599999904632568,0.23000000417232513,0.019999999552965164,--,--,--,--,--,--,--,--,--,--,--,--,0.73,0.12,0.05,0.1,1.0.0,DR8


In [3]:
# Load FP catalog
FP = Table.read('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/FP_Targets_SSL.csv')
FP

targetid,survey,program,healpix,brickid,brickname,brick_objid,release,target_ra,target_dec,std_fiber_ra,std_fiber_dec,z,zerr,zwarn,deltachi2,spectype,subtype,ra,dec,ra_ivar,dec_ivar,morphtype,sersic,sersic_ivar,flux_g,flux_r,flux_z,flux_w1,flux_w2,flux_ivar_g,flux_ivar_r,flux_ivar_z,flux_ivar_w1,flux_ivar_w2,mw_transmission_g,mw_transmission_r,mw_transmission_z,mw_transmission_w1,mw_transmission_w2,fiberflux_g,fiberflux_r,fiberflux_z,fibertotflux_g,fibertotflux_r,fibertotflux_z,fracflux_g,fracflux_r,fracflux_z,fracin_g,fracin_r,fracin_z,fracmasked_g,fracmasked_r,fracmasked_z,ebv,gaia_phot_g_mean_mag,gaia_phot_g_mean_flux_over_error,maskbits,shape_e1,shape_e2,shape_e1_ivar,shape_e2_ivar,shape_r,shape_r_ivar,ls_id,nobs_g,nobs_r,nobs_z,npixels,dchisq_psf,dchisq_rex,dchisq_dev,dchisq_exp,dchisq_ser,coadd_exptime,coadd_numexp,coadd_numnight,coadd_numtile,zcat_primary,cmx_target,desi_target,bgs_target,mws_target,scnd_target,sv1_desi_target,sv1_bgs_target,sv1_mws_target,sv1_scnd_target,sv2_desi_target,sv2_bgs_target,sv2_mws_target,sv2_scnd_target,sv3_desi_target,sv3_bgs_target,sv3_mws_target,sv3_scnd_target,circ_radius,BA_ratio,pos_angle,shape_e1_err,shape_e2_err,BA_ratio_err,circ_radius_err,shape_epsilon,shape_epsilon_err,mag_g,mag_err_g,fiber_mag_g,mag_z,mag_err_z,fiber_mag_z,mag_r,mag_err_r,fiber_mag_r,dist_L,dist_L_err,absmag_r,in_south,filename,zcmb,ppxf_vdisp,ppxf_vdisp_err,ppxf_chi2,ppxf_corrected_err,ppxf_snr,dnr_snr,ppxf_dof,Train-Test_Split,fullpath
int64,str7,str6,int64,int64,str8,int64,int64,float64,float64,float64,float64,float64,float64,int64,float64,str6,int64,float64,float64,float64,float64,str3,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,int64,int64,int64,str1,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str5,str70,float64,float64,float64,float64,float64,float64,float64,float64,str4,str107
2398569927540753,main,bright,4188,389714,1433p102,4709,9010,143.41941588269418,10.132311652451484,0.0,0.0,0.0098835598796803,6.924632027465498e-05,0,49.92314263700246,GALAXY,--,143.41945945273935,10.132307679023883,280387000000.0,234928420000.0,SER,2.7763934,3281.5857,30.822884,62.963234,99.07233,-26.025707,-30.247889,38.111427,8.318946,2.0942228,0.6721521,0.17561215,0.91230106,0.94004464,0.96600753,0.99475914,0.99677813,1.9900682,4.0651984,6.396569,2.0413795,4.18489,6.624818,0.15795921,0.17750928,0.22452308,0.9297771,0.93101,0.98966026,0.00034343117,0.00010865969,0.005893525,0.031006388,0.0,0.0,4096,0.057115875,-0.004976765,108960.59,39629.953,5.479398,2329.3965,9906625306563173,3,3,3,7868,0.0,0.0,0.0,0.0,0.0,841.0511,1,1,1,t,0,4611686018427387904,0,0,35184372088832,0,0,0,0,0,0,0,0,0,0,0,0,5.173761623854078,0.8915529400986123,-2.489931614654116,0.0030294603788807,0.005023289655163,1.7931420005594064e-05,0.0207195265089186,0.0573322890427449,1.002322956888176e-05,18.678162269054205,0.0052054900711216,21.653175548634504,17.472570337041795,0.0073154494395256,20.447583556861165,17.935153626881146,0.0056201980265729,20.910166828756825,29856334.95565034,2047.207842047456,-14.440028845149476,True,loa/healpix/main/bright/41/4188/redrock-main-bright-4188.fits,0.0109665920029964,162.95208725275575,137.16716704626725,0.3226116994890419,77.90946563375547,0.6457295706664297,1.1639210713561363,2560.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healpix/main/bright/41/4188/redrock-main-bright-4188.fits
2411764067074053,main,dark,4188,389714,1433p102,4709,9010,143

In [61]:
# Create SkyCoord objects
galzoo_coords = SkyCoord(ra=galaxyzoo["ra"] * u.deg,
                     dec=galaxyzoo["dec"] * u.deg)

fp_coords = SkyCoord(ra=FP["ra"] * u.deg,
                      dec=FP["dec"] * u.deg)

# Do a cone search (cross-match) within 1 arcseconds
idx_fp, idx_galzoo, sep2d, _ = galzoo_coords.search_around_sky(fp_coords, 1 * u.arcsec)

# This gives you the indices of matches from both catalogs
print(f"Number of matched targets: {len(idx_fp)}")
print(f"Number of matched targets: {len(idx_galzoo)}")

Number of matched targets: 154969
Number of matched targets: 154969


In [65]:
# --- Build candidate-pair table (still one row per candidate match) ---
matches = Table()

# FP fields
matches["fp_targetid"] = FP["targetid"][idx_fp]
matches["fp_ra"]       = FP["ra"][idx_fp]
matches["fp_dec"]      = FP["dec"][idx_fp]

# Galaxy Zoo fields
matches["dr8_id"] = galaxyzoo["dr8_id"][idx_galzoo]
matches["gz_ra"]  = galaxyzoo["ra"][idx_galzoo]
matches["gz_dec"] = galaxyzoo["dec"][idx_galzoo]

# Add Galaxy Zoo vote fractions (needed for morphology classification)
vote_columns = [
    "has-spiral-arms_yes_fraction",
    "smooth-or-featured_smooth_fraction",
    "smooth-or-featured_featured-or-disk_fraction",
    "smooth-or-featured_artifact_fraction",
    "merging_major-disturbance_fraction",
    "merging_minor-disturbance_fraction",
    "merging_merger_fraction",
]
for col in vote_columns:
    matches[col] = galaxyzoo[col][idx_galzoo]

# Separation
matches["sep_arcsec"] = sep2d.arcsec

# --- Greedy one-to-one selection (smallest separation wins; no repeats on either side) ---
order = np.argsort(matches["sep_arcsec"])

used_fp = set()
used_gz = set()
keep = []

for i in order:
    fp_id = matches["fp_targetid"][i]
    gz_id = matches["dr8_id"][i]
    if (fp_id not in used_fp) and (gz_id not in used_gz):
        keep.append(i)
        used_fp.add(fp_id)
        used_gz.add(gz_id)

matches_1to1 = matches[keep]
matches_1to1.sort("sep_arcsec")

matches_1to1

fp_targetid,fp_ra,fp_dec,dr8_id,gz_ra,gz_dec,has-spiral-arms_yes_fraction,smooth-or-featured_smooth_fraction,smooth-or-featured_featured-or-disk_fraction,smooth-or-featured_artifact_fraction,merging_major-disturbance_fraction,merging_minor-disturbance_fraction,merging_merger_fraction,sep_arcsec
int64,float64,float64,str12,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
39627532290492361,358.0726659999005,-10.684326999434957,270182_3194,358.072666,-10.684327,--,0.74,0.14,0.11,0.1,0.22,0.01,2.0644099146914896e-06
2842376347320329,204.7980510005146,-6.318998998355204,295234_1145,204.798051,-6.318999,--,0.52,0.14,0.34,0.06,0.12,0.21,6.200949008550869e-06
39627775207802521,152.58498600083666,-0.4817899984679515,328098_2859,152.584986,-0.48179,--,0.74,0.16,0.1,0.05,0.13,0.01,6.284195180989364e-06
39628332018434898,1.6823129977178477,23.369285001306807,460852_3871,1.682313,23.369285,--,0.79,0.13,0.08,0.03,0.09,0.01,8.88881072889495e-06
2305843018621522677,225.53148399797067,23.13789800317171,461676_283,225.531484,23.137898,--,0.73,0.21,0.05,0.02,0.07,0.01,1.324784280523956e-05
39633365887880634,250.73685199339164,58.24646800184644,612444_2214,250.736852,58.246468,--,0.71,0.17,0.12,0.05,0.12,0.02,1.4175083066145288e-05
39628072458126268,337.6073220024723,11.709640996676708,398968_1583,337.607322,11.709641,--,0.75,0.17,0.08,0.02,0.07,0.01,1.480153473025379e-05
39633461048250261,256.09521899910106,66.80124699548394,635132_2916,256.095219,66.801247,--,0.81,0.08,0.11,0.03,0.08,0.02,1.6307694543765215e-05
39627756660592734,127.1786749966846,-1.1561849968108018,323676_4723,127.178675,-1.156185,--,0.76,0.1,0.14,0.04,0.09,0.01,1.655934921258623e-05


In [66]:
print("FP duplicates:", len(matches_1to1) != len(np.unique(matches_1to1["fp_targetid"])))
print("GZ duplicates:", len(matches_1to1) != len(np.unique(matches_1to1["dr8_id"])))

FP duplicates: False
GZ duplicates: False


In [67]:
# Very basic classification scheme
def classify_morphology(row):
    # Key vote fractions
    spiral = row.get('has-spiral-arms_yes_fraction', 0)
    smooth = row.get('smooth-or-featured_smooth_fraction', 0)
    featured = row.get('smooth-or-featured_featured-or-disk_fraction', 0)
    artifact = row.get('smooth-or-featured_artifact_fraction', 0)

    disturbed = row.get('merging_major-disturbance_fraction', 0) + \
                row.get('merging_minor-disturbance_fraction', 0) + \
                row.get('merging_merger_fraction', 0)

    # Classification Rules based on DESI GZ decision tree
    # 1. Artifact or strongly disturbed → Irregular
    if artifact > 0.3 or disturbed > 0.3:
        return 'Irregular'

    # 2. Clear spiral arms → Spiral
    if spiral > 0.5:
        return 'Spiral'

    # 3. Smooth galaxy (no features) → Elliptical
    if smooth > 0.5:
        return 'Elliptical'

    # 4. Disk-like, no arms, not smooth → Lenticular
    if spiral < 0.3 and smooth < 0.5 and featured > 0.3:
        return 'Lenticular'

    # 5. unclassifiable
    return 'Unclassified'

In [68]:
# Apply classification row-by-row
morph_classes = [
    classify_morphology(dict(row))
    for row in matches_1to1
]

# Add as a new column
matches_1to1["morphology"] = morph_classes

In [69]:
types, counts = np.unique(matches_1to1["morphology"], return_counts=True)

for t, c in zip(types, counts):
    print(f"{t}: {c}")

Elliptical: 96951
Irregular: 22031
Lenticular: 454
Spiral: 3215
Unclassified: 7519


In [70]:
mask = matches_1to1["morphology"] != "Unclassified"
matches_clean = matches_1to1[mask]

matches_clean

fp_targetid,fp_ra,fp_dec,dr8_id,gz_ra,gz_dec,has-spiral-arms_yes_fraction,smooth-or-featured_smooth_fraction,smooth-or-featured_featured-or-disk_fraction,smooth-or-featured_artifact_fraction,merging_major-disturbance_fraction,merging_minor-disturbance_fraction,merging_merger_fraction,sep_arcsec,morphology
int64,float64,float64,str12,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str12
39627532290492361,358.0726659999005,-10.684326999434957,270182_3194,358.072666,-10.684327,--,0.74,0.14,0.11,0.1,0.22,0.01,2.0644099146914896e-06,Irregular
2842376347320329,204.7980510005146,-6.318998998355204,295234_1145,204.798051,-6.318999,--,0.52,0.14,0.34,0.06,0.12,0.21,6.200949008550869e-06,Irregular
39627775207802521,152.58498600083666,-0.4817899984679515,328098_2859,152.584986,-0.48179,--,0.74,0.16,0.1,0.05,0.13,0.01,6.284195180989364e-06,Elliptical
39628332018434898,1.6823129977178477,23.369285001306807,460852_3871,1.682313,23.369285,--,0.79,0.13,0.08,0.03,0.09,0.01,8.88881072889495e-06,Elliptical
2305843018621522677,225.53148399797067,23.13789800317171,461676_283,225.531484,23.137898,--,0.73,0.21,0.05,0.02,0.07,0.01,1.324784280523956e-05,Elliptical
39633365887880634,250.73685199339164,58.24646800184644,612444_2214,250.736852,58.246468,--,0.71,0.17,0.12,0.05,0.12,0.02,1.4175083066145288e-05,Elliptical
39628072458126268,337.6073220024723,11.709640996676708,398968_1583,337.607322,11.709641,--,0.75,0.17,0.08,0.02,0.07,0.01,1.480153473025379e-05,Elliptical
39633461048250261,256.09521899910106,66.80124699548394,635132_2916,256.095219,66.801247,--,0.81,0.08,0.11,0.03,0.08,0.02,1.6307694543765215e-05,Elliptical
39627756660592734,127.1786749966846,-1.1561849968108018,323676_4723,127.178675,-1.156185,--,0.76,0.1,0.14,0.04,0.09,0.01,1.655934921258623e-05,Elliptical


In [71]:
np.unique(matches_clean["morphology"], return_counts=True)

(<Column name='morphology' dtype='str12' length=4>
 Elliptical
  Irregular
 Lenticular
     Spiral,
 array([96951, 22031,   454,  3215]))

In [73]:
matches_clean.write('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/Generating_Anchor_Points/gz_fp_comparison.csv', format="csv", overwrite=True)